In [79]:
import torch
from torch import nn
dropout=0.5
net=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Dropout(dropout),
                  nn.Linear(32,128),nn.ReLU(),nn.Dropout(dropout),
                  nn.Linear(128,8))
X=torch.rand(2,20)
net(X)

tensor([[ 0.1307, -0.1294,  0.0029,  0.0984,  0.0991, -0.0520,  0.0868,  0.2809],
        [ 0.0143, -0.0739,  0.1565, -0.3161,  0.1085,  0.0638, -0.0134,  0.0811]],
       grad_fn=<AddmmBackward0>)

In [80]:
class MY_DIY_Net(nn.Module):
    def __init__(self, input_size, output_size):
        super(MY_DIY_Net, self).__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.relu = nn.ReLU()
        self.linear1 = nn.Linear(input_size, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64, output_size)
    
    def forward(self, x):
        H1=self.relu(self.linear1(x))
        H1=nn.Dropout(0.5)(H1)
        H2=self.relu(self.linear2(H1))
        H2=nn.Dropout(0.5)(H2)
        out=self.linear3(H2)
        return out

# create an instance of the network
net = MY_DIY_Net(input_size=20, output_size=10)


In [81]:
net(X)

tensor([[-0.2178, -0.0826,  0.3225,  0.0348, -0.0370,  0.0441,  0.1788,  0.0677,
          0.0169,  0.1723],
        [-0.0970, -0.4508,  0.2241,  0.0729, -0.2493, -0.1282,  0.1930,  0.1244,
          0.1885,  0.1824]], grad_fn=<AddmmBackward0>)

In [82]:
class FixedNet(nn.Module):
    def __init__(self):
        super(FixedNet, self).__init__()
        self.rand_weight = nn.Parameter(torch.rand(10, 10),requires_grad= False)
        self.linear= nn.Linear(20, 10)
        self.relu= nn.ReLU()

    def forward(self, x):
        H1= self.linear(x)
        H2=self.relu(torch.mm(H1, self.rand_weight)+2)#等价于一个参数不变的等长隐藏层
        out= self.relu(H2)
        while out.abs().sum()>0.5:
            res=out.abs().sum()-0.5
            if res.abs()<0.45:
              out= out+ res
            else:
                out=out/2
            print('out sum',out.abs().sum())
        return out.sum()
net= FixedNet()
net(X)

out sum tensor(24.1164, grad_fn=<SumBackward0>)
out sum tensor(12.0582, grad_fn=<SumBackward0>)
out sum tensor(6.0291, grad_fn=<SumBackward0>)
out sum tensor(3.0145, grad_fn=<SumBackward0>)
out sum tensor(1.5073, grad_fn=<SumBackward0>)
out sum tensor(0.7536, grad_fn=<SumBackward0>)
out sum tensor(5.8264, grad_fn=<SumBackward0>)
out sum tensor(2.9132, grad_fn=<SumBackward0>)
out sum tensor(1.4566, grad_fn=<SumBackward0>)
out sum tensor(0.7283, grad_fn=<SumBackward0>)
out sum tensor(5.2942, grad_fn=<SumBackward0>)
out sum tensor(2.6471, grad_fn=<SumBackward0>)
out sum tensor(1.3236, grad_fn=<SumBackward0>)
out sum tensor(0.6618, grad_fn=<SumBackward0>)
out sum tensor(3.8973, grad_fn=<SumBackward0>)
out sum tensor(1.9487, grad_fn=<SumBackward0>)
out sum tensor(0.9743, grad_fn=<SumBackward0>)
out sum tensor(0.4872, grad_fn=<SumBackward0>)


tensor(0.4872, grad_fn=<SumBackward0>)

In [83]:
net=nn.Sequential(MY_DIY_Net(input_size=20,output_size=10),nn.Linear(10,20),FixedNet())
net(X)

out sum tensor(22.9506, grad_fn=<SumBackward0>)
out sum tensor(11.4753, grad_fn=<SumBackward0>)
out sum tensor(5.7377, grad_fn=<SumBackward0>)
out sum tensor(2.8688, grad_fn=<SumBackward0>)
out sum tensor(1.4344, grad_fn=<SumBackward0>)
out sum tensor(0.7172, grad_fn=<SumBackward0>)
out sum tensor(5.0613, grad_fn=<SumBackward0>)
out sum tensor(2.5307, grad_fn=<SumBackward0>)
out sum tensor(1.2653, grad_fn=<SumBackward0>)
out sum tensor(0.6327, grad_fn=<SumBackward0>)
out sum tensor(3.2860, grad_fn=<SumBackward0>)
out sum tensor(1.6430, grad_fn=<SumBackward0>)
out sum tensor(0.8215, grad_fn=<SumBackward0>)
out sum tensor(7.2515, grad_fn=<SumBackward0>)
out sum tensor(3.6258, grad_fn=<SumBackward0>)
out sum tensor(1.8129, grad_fn=<SumBackward0>)
out sum tensor(0.9064, grad_fn=<SumBackward0>)
out sum tensor(9.0353, grad_fn=<SumBackward0>)
out sum tensor(4.5176, grad_fn=<SumBackward0>)
out sum tensor(2.2588, grad_fn=<SumBackward0>)
out sum tensor(1.1294, grad_fn=<SumBackward0>)
out sum ten

tensor(0.4982, grad_fn=<SumBackward0>)

In [84]:
class Mix_Net(nn.Module):
    def __init__(self):
        super(Mix_Net, self).__init__()
        self.net1= MY_DIY_Net(input_size=20, output_size=10)
        self.net2= FixedNet()
        self.connet= nn.Linear(10,20)

    def forward(self, x):
        H1= self.net1(x)
        H2= self.connet(H1)
        out= self.net2(H2)
        return out
net= Mix_Net()
net(X)

out sum tensor(23.0961, grad_fn=<SumBackward0>)
out sum tensor(11.5481, grad_fn=<SumBackward0>)
out sum tensor(5.7740, grad_fn=<SumBackward0>)
out sum tensor(2.8870, grad_fn=<SumBackward0>)
out sum tensor(1.4435, grad_fn=<SumBackward0>)
out sum tensor(0.7218, grad_fn=<SumBackward0>)
out sum tensor(5.1568, grad_fn=<SumBackward0>)
out sum tensor(2.5784, grad_fn=<SumBackward0>)
out sum tensor(1.2892, grad_fn=<SumBackward0>)
out sum tensor(0.6446, grad_fn=<SumBackward0>)
out sum tensor(3.5367, grad_fn=<SumBackward0>)
out sum tensor(1.7683, grad_fn=<SumBackward0>)
out sum tensor(0.8842, grad_fn=<SumBackward0>)
out sum tensor(8.5677, grad_fn=<SumBackward0>)
out sum tensor(4.2838, grad_fn=<SumBackward0>)
out sum tensor(2.1419, grad_fn=<SumBackward0>)
out sum tensor(1.0710, grad_fn=<SumBackward0>)
out sum tensor(0.5355, grad_fn=<SumBackward0>)
out sum tensor(1.2451, grad_fn=<SumBackward0>)
out sum tensor(0.6225, grad_fn=<SumBackward0>)
out sum tensor(3.0731, grad_fn=<SumBackward0>)
out sum ten

tensor(0.4855, grad_fn=<SumBackward0>)

In [85]:
class Combine_block(nn.Module):
    def __init__(self):
        super(Combine_block, self).__init__()
        self.net1= nn.Linear(20,33)
        self.net2= nn.Linear(20,66)
        
        self.relu= nn.ReLU()


    def forward(self, x):
        H1_1= self.net1(x)
        H1_2= self.net2(x)
        out= torch.cat((H1_1, H1_2), dim=1)
        out= self.relu(out)
        out=nn.Linear(out.shape[1], 20)(out)
        return out

net=Combine_block()
net(X)
print(net)

Combine_block(
  (net1): Linear(in_features=20, out_features=33, bias=True)
  (net2): Linear(in_features=20, out_features=66, bias=True)
  (relu): ReLU()
)


In [86]:


net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[ 0.0397],
        [-0.1311]], grad_fn=<AddmmBackward0>)

In [87]:
print(net[0].weight)
print(net[0].bias)
print(net[2].state_dict())

Parameter containing:
tensor([[ 0.2497, -0.2164,  0.3980,  0.3689],
        [-0.1357, -0.3825,  0.1670, -0.2347],
        [-0.1926, -0.0935, -0.1348,  0.0125],
        [ 0.1521, -0.0402,  0.1464, -0.4843],
        [ 0.2716, -0.2974,  0.3458, -0.0732],
        [-0.4708, -0.0448, -0.4559, -0.4262],
        [-0.0987,  0.2750, -0.1667, -0.0110],
        [-0.1175, -0.0038,  0.4951,  0.3251]], requires_grad=True)
Parameter containing:
tensor([ 0.4298, -0.4678,  0.3765,  0.0395,  0.0981, -0.0475,  0.3567, -0.0211],
       requires_grad=True)
OrderedDict([('weight', tensor([[ 0.0272, -0.2045, -0.2874,  0.1849,  0.2756, -0.0586,  0.0758,  0.2159]])), ('bias', tensor([-0.1848]))])


In [88]:
print(*[(name, param.shape) for name, param in net[2].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([1, 8])) ('bias', torch.Size([1]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [89]:
print(net.state_dict()['2.weight'].data)

tensor([[ 0.0272, -0.2045, -0.2874,  0.1849,  0.2756, -0.0586,  0.0758,  0.2159]])


In [90]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net
rgnet=nn.Sequential(block2(),nn.Linear(4,1))
print(rgnet)

rgnet[0][0][0].weight

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


Parameter containing:
tensor([[-0.2677,  0.2878,  0.0548,  0.4176],
        [ 0.1570,  0.3061, -0.1496, -0.1863],
        [ 0.0408,  0.4162, -0.0826, -0.2411],
        [-0.1398, -0.3679,  0.4228, -0.2413],
        [ 0.0164,  0.1439, -0.3586,  0.3122],
        [-0.1370, -0.1387,  0.0691, -0.2839],
        [ 0.1158, -0.4108,  0.3879,  0.3598],
        [-0.2540, -0.4612, -0.1654,  0.4120]], requires_grad=True)

In [91]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)
rgnet.apply(init_weights)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

稠密层设计

In [92]:
shared=nn.Linear(10,10)
net=nn.Sequential(nn.Linear(4,10),nn.ReLU(),
                  shared,nn.ReLU(),
                  shared,nn.ReLU(),
                  nn.Linear(10,16),nn.ReLU(),
                  nn.Linear(16,1))
net(X)
net[2].weight.data_ptr()#返回地址
net[2].weight.data#返回参数
print(net[2].weight.data_ptr()==net[4].weight.data_ptr())#判断函数权重内部是否相等

True


延后初始化:输入数据的形状不确定，延后初始化允许模型根据实际输入动态调整结构，无需提前指定参数形状;对于包含大量参数的大型模型，延后初始化以避免在模型构建时占用过多内存资源，直到真正访问这些参数时才动态分配。


In [93]:
import torch
import torch.nn as nn

class LazyLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = None  # 不在初始化时设置 Linear 层

    def forward(self, data):
        if self.linear is None:
            self.linear = nn.Linear(in_features=data.shape[-1], out_features=10)  # 根据输入的形状初始化 Linear 层
        return self.linear(data)


if __name__ == '__main__':
    model = LazyLinear()
    print(f'初始化前：\n{model = }')
    print(f'初始化并前向传播：\n{model(torch.randn(5, 20)).shape = }')
    print(f'初始化后：\n{model = }')
    
class LazyLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.LazyLinear(out_features=10)  # 使用延后初始化线性层

    def forward(self, x):
        return self.linear(x)


if __name__ == '__main__':
    # import warnings; warnings.filterwarnings("ignore", category=UserWarning)

    model = LazyLinear()
    print(f'初始化前：\n{model = }')
    print(f'初始化并前向传播：\n{model(torch.randn(5, 20)).shape = }')
    print(f'初始化后：\n{model = }')

初始化前：
model = LazyLinear()
初始化并前向传播：
model(torch.randn(5, 20)).shape = torch.Size([5, 10])
初始化后：
model = LazyLinear(
  (linear): Linear(in_features=20, out_features=10, bias=True)
)
初始化前：
model = LazyLinear(
  (linear): LazyLinear(in_features=0, out_features=10, bias=True)
)
初始化并前向传播：
model(torch.randn(5, 20)).shape = torch.Size([5, 10])
初始化后：
model = LazyLinear(
  (linear): Linear(in_features=20, out_features=10, bias=True)
)


In [94]:
class Nonething(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, X):
        return X-X.mean()

net=Nonething()

test=torch.randn(1,10)
net(test)

tensor([[ 0.8882,  1.5552, -0.3906,  0.2727, -0.5031,  0.1376, -1.1193, -0.7095,
          0.0189, -0.1501]])

In [95]:
net=nn.Sequential(nn.Linear(10,16),Nonething())
test1=torch.tensor([[1.0]*10])
net(test1).mean()

tensor(-2.2352e-08, grad_fn=<MeanBackward0>)

In [98]:
class My_Layer(nn.Module):
    def __init__(self, in_units, units):
        super(My_Layer, self).__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units))
        self.activation = nn.ReLU()
    def forward(self, X):
        out= self.activation(torch.matmul(X, self.weight)+ self.bias)
        return out
       

In [112]:
My_Layer(3,4).weight.data

tensor([[ 1.4771,  1.9139, -1.0021, -0.3382],
        [-0.7166, -0.4027,  1.7545,  1.3233],
        [ 0.3082, -2.8042, -0.5322, -1.6111]])

tensor([[0.7233, 0.4423, 0.7302],
        [0.4166, 0.9017, 0.0487],
        [0.5508, 0.4179, 0.4143]])